In [1]:
import numpy as np
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF
from sklearn.linear_model import Ridge, LogisticRegression
from sklearn.neighbors import KNeighborsRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR
from sklearn.preprocessing import StandardScaler
from sklearn.base import clone
from scipy.spatial import Voronoi
from scipy.stats import norm
import warnings
warnings.filterwarnings('ignore')

# ============================================
# HELPER FUNCTIONS
# ============================================

def loocv_score(model, X, y):
    n = len(X)
    scores = []
    for i in range(n):
        X_train = np.delete(X, i, axis=0)
        y_train = np.delete(y, i, axis=0)
        X_test = X[i].reshape(1, -1)
        y_test = y[i]
        model_copy = clone(model)
        model_copy.fit(X_train, y_train)
        y_pred = model_copy.predict(X_test)[0]
        scores.append(-(y_pred - y_test)**2)
    return np.mean(scores)

def logistic_regression_analysis(X_train, y_train, threshold='median'):
    if threshold == 'median':
        thresh_val = np.median(y_train)
    else:
        thresh_val = threshold
    y_binary = (y_train > thresh_val).astype(int)
    n_class1 = np.sum(y_binary)
    n_class0 = len(y_binary) - n_class1
    if n_class1 == 0 or n_class0 == 0:
        return None, 0, "Only one class present"
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X_train)
    lr = LogisticRegression(max_iter=1000, random_state=42)
    lr.fit(X_scaled, y_binary)
    coefficients = lr.coef_[0]
    accuracy = np.mean(lr.predict(X_scaled) == y_binary)
    return coefficients, accuracy, None

def get_model_suggestion_ei(model, X_train, y_train, dim, xi=0.01, n_candidates=10000):
    if isinstance(model, GaussianProcessRegressor):
        candidates = np.random.rand(n_candidates, dim)
        mean, std = model.predict(candidates, return_std=True)
        f_best = np.max(y_train)
        improvement = mean - f_best - xi
        Z = improvement / (std + 1e-9)
        ei_scores = improvement * norm.cdf(Z) + std * norm.pdf(Z)
        ei_scores[std < 1e-9] = 0
        best_idx = np.argmax(ei_scores)
        return candidates[best_idx]
    else:
        candidates = np.random.rand(n_candidates, dim)
        predictions = model.predict(candidates)
        best_idx = np.argmax(predictions)
        return candidates[best_idx]

def check_boundaries(point, dim, margin=0.001):
    for val in point:
        if val < margin or val > 1 - margin:
            return True
    return False

def y_weighted_centroid(X_train, y_train, top_k=3):
    sorted_idx = np.argsort(y_train)[::-1]
    top_idx = sorted_idx[:top_k]
    top_X = X_train[top_idx]
    top_y = y_train[top_idx]
    if np.min(top_y) < 0:
        weights = top_y - np.min(top_y) + 0.001
    else:
        weights = top_y
    centroid = np.average(top_X, axis=0, weights=weights)
    return np.round(centroid, 6)

def voronoi_next_point(X_existing, bounds=(0, 1), n_random=10000):
    boundary_points = np.array([
        [bounds[0], bounds[0]], [bounds[0], bounds[1]],
        [bounds[1], bounds[0]], [bounds[1], bounds[1]],
        [bounds[0], 0.5], [bounds[1], 0.5], [0.5, bounds[0]], [0.5, bounds[1]]
    ])
    all_points = np.vstack([X_existing, boundary_points])
    try:
        vor = Voronoi(all_points)
        best_center = None
        best_radius = -np.inf
        for i, region_idx in enumerate(vor.point_region):
            if region_idx >= len(vor.regions) or len(vor.regions[region_idx]) == 0:
                continue
            vertices = vor.vertices[vor.regions[region_idx]]
            if len(vertices) == 0:
                continue
            if np.any(vertices[:, 0] < bounds[0]) or np.any(vertices[:, 0] > bounds[1]):
                continue
            if np.any(vertices[:, 1] < bounds[0]) or np.any(vertices[:, 1] > bounds[1]):
                continue
            center = np.mean(vertices, axis=0)
            distances = np.linalg.norm(vertices - center, axis=1)
            radius = np.max(distances)
            if radius > best_radius:
                best_radius = radius
                best_center = center
        if best_center is None:
            raise ValueError("Voronoi failed")
    except Exception:
        candidates = np.random.rand(n_random, 2)
        min_distances = []
        for cand in candidates:
            dists = np.linalg.norm(X_existing - cand, axis=1)
            min_distances.append(np.min(dists))
        best_idx = np.argmax(min_distances)
        best_center = candidates[best_idx]
    best_center = np.clip(best_center, bounds[0], bounds[1])
    return np.round(best_center, 6)

def tune_model(model_name, X_train, y_train, dim, scaler=None):
    if model_name == 'Ridge':
        param_grid = {'alpha': [0.01, 0.1, 1.0, 10.0]}
        base_model = Ridge()
    elif model_name == 'KNN':
        param_grid = {'n_neighbors': [3, 5, 7, 9]}
        base_model = KNeighborsRegressor()
    elif model_name == 'RandomForest':
        param_grid = {'n_estimators': [50, 100], 'max_depth': [5, 10, None]}
        base_model = RandomForestRegressor(random_state=42)
    elif model_name == 'SVR':
        param_grid = {'C': [0.1, 1.0, 10.0, 100.0], 'epsilon': [0.01, 0.1, 0.5]}
        base_model = SVR(kernel='rbf')
    elif model_name == 'GP':
        param_grid = {'length_scale': [0.1, 0.5, 1.0, 2.0]}
        base_model = GaussianProcessRegressor(normalize_y=False, random_state=42)
    else:
        return None, -np.inf
    best_score = -np.inf
    best_model = None
    if model_name == 'GP':
        for ls in param_grid['length_scale']:
            try:
                model = GaussianProcessRegressor(
                    kernel=RBF(length_scale=ls),
                    alpha=1e-10,
                    normalize_y=False,
                    random_state=42
                )
                if scaler:
                    X_use = scaler.transform(X_train)
                else:
                    X_use = X_train
                model.fit(X_use, y_train)
                score = loocv_score(model, X_use, y_train)
                if score > best_score:
                    best_score = score
                    best_model = model
            except:
                continue
    else:
        for params in [dict(zip(param_grid.keys(), vals)) for vals in zip(*param_grid.values())]:
            try:
                model = base_model.__class__(**params)
                if scaler:
                    X_use = scaler.transform(X_train)
                else:
                    X_use = X_train
                model.fit(X_use, y_train)
                score = loocv_score(model, X_use, y_train)
                if score > best_score:
                    best_score = score
                    best_model = model
            except:
                continue
    return best_model, best_score

def sanity_check(suggestion, X_train, y_train, dim, best_point, best_output, trust_radius=0.15):
    warnings_list = []
    checked = suggestion.copy()
    distance_to_best = np.linalg.norm(suggestion - best_point)
    if distance_to_best > trust_radius:
        warnings_list.append(f"Point is {distance_to_best:.3f} from best (radius={trust_radius})")
        direction = suggestion - best_point
        direction = direction / (distance_to_best + 1e-9)
        checked = best_point + direction * trust_radius
        checked = np.clip(checked, 0, 1)
        warnings_list.append(f"  → Pulled back to {checked}")
    for i, val in enumerate(checked):
        if val < 0.01:
            warnings_list.append(f"Dimension {i+1} at boundary {val:.4f} (very low)")
        if val > 0.99:
            warnings_list.append(f"Dimension {i+1} at boundary {val:.4f} (very high)")
    distances = np.linalg.norm(X_train - checked, axis=1)
    nearest_idx = np.argmin(distances)
    nearest_dist = distances[nearest_idx]
    nearest_output = y_train[nearest_idx]
    if nearest_dist < 0.05 and nearest_output < np.percentile(y_train, 25):
        warnings_list.append(f"Very close ({nearest_dist:.3f}) to poor output {nearest_output:.4f}")
    return np.round(checked, 6), warnings_list

def select_next_query(X_train, y_train, dim, function_name):
    lr_coeffs, lr_accuracy, lr_warning = logistic_regression_analysis(X_train, y_train)
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X_train)
    best_idx = np.argmax(y_train)
    best_point = X_train[best_idx]
    best_output = y_train[best_idx]
    model_names = ['Ridge', 'KNN', 'RandomForest', 'SVR', 'GP']
    baseline_score = -np.var(y_train)
    print(f"\n  {function_name}:")
    print(f"  Baseline LOOCV score: {baseline_score:.4f}")
    print(f"  Current best: {best_output:.4f} at {best_point}")
    if lr_coeffs is not None:
        print(f"\n  Logistic Regression Diagnostic (threshold=median):")
        print(f"    Accuracy: {lr_accuracy:.4f}")
        print(f"    Coefficients (direction: + = higher Y, - = lower Y):")
        for i, coeff in enumerate(lr_coeffs):
            direction = "↑" if coeff > 0.1 else "↓" if coeff < -0.1 else "~"
            print(f"      x{i+1}: {coeff:+.4f} {direction}")
        if lr_accuracy > 0.8:
            print(f"    → High accuracy suggests approximately linear pattern")
        elif lr_accuracy < 0.6:
            print(f"    → Low accuracy suggests non-linear pattern")
    else:
        print(f"\n  Logistic Regression Diagnostic: {lr_warning}")
    best_model = None
    best_score = -np.inf
    best_suggestion = None
    best_model_name = None
    for name in model_names:
        try:
            print(f"\n    Tuning {name}...")
            if name == 'GP':
                model, score = tune_model(name, X_train, y_train, dim, scaler=None)
                if model:
                    suggestion = get_model_suggestion_ei(model, X_train, y_train, dim)
                else:
                    continue
            else:
                model, score = tune_model(name, X_scaled, y_train, dim, scaler=scaler)
                if model:
                    suggestion = get_model_suggestion_ei(model, X_scaled, y_train, dim)
                    suggestion = scaler.inverse_transform(suggestion.reshape(1, -1))[0]
                else:
                    continue
            suggestion = np.round(suggestion, 6)
            at_boundary = check_boundaries(suggestion, dim)
            beats_baseline = score > baseline_score
            print(f"      {name}: LOOCV={score:.4f}, beats_baseline={beats_baseline}, at_boundary={at_boundary}")
            print(f"      Suggestion: {suggestion}")
            if beats_baseline and not at_boundary and score > best_score:
                best_score = score
                best_model = model
                best_suggestion = suggestion
                best_model_name = name
        except Exception as e:
            print(f"    {name}: FAILED - {str(e)[:50]}")
            continue
    if best_suggestion is None:
        print(f"\n    No model beat baseline. Using Y-weighted centroid.")
        best_suggestion = y_weighted_centroid(X_train, y_train)
        best_model_name = "Centroid"
        best_score = -np.inf
    print(f"\n    Running sanity checks on {best_model_name} suggestion...")
    final_suggestion, warnings = sanity_check(
        best_suggestion, X_train, y_train, dim, best_point, best_output, trust_radius=0.15
    )
    if warnings:
        print(f"    WARNINGS:")
        for w in warnings:
            print(f"      {w}")
        print(f"    Original: {best_suggestion}")
        print(f"    Final: {final_suggestion}")
    else:
        print(f"    No warnings. Final: {final_suggestion}")
    print(f"\n  SELECTED: {best_model_name} (score={best_score:.4f})")
    return final_suggestion

# ============================================
# FUNCTION 1 (2D) - Local exploitation (signal found)
# ============================================
X1 = np.array([
    [0.31940389, 0.76295937],
    [0.57432921, 0.87989810],
    [0.73102363, 0.73299988],
    [0.84035342, 0.26473161],
    [0.65011406, 0.68152635],
    [0.41043714, 0.14755430],
    [0.31269116, 0.07872278],
    [0.68341817, 0.86105746],
    [0.08250725, 0.40348751],
    [0.88388983, 0.58225397],
    [0.19879800, 0.72650400],
    [0.44544600, 0.30339500]   
])
y1 = np.array([1.32267704e-079, 1.03307824e-046, 7.71087511e-016, 3.34177101e-124,
               -3.60606264e-003, -2.15924904e-054, -2.08909327e-091, 2.53500115e-040,
               3.60677119e-081, 6.22985647e-048, -6.30554291e-100, 4.81307631e-13])

# Find best point
best_idx = np.argmax(y1)
best_point = X1[best_idx]  # (0.445446, 0.303395)

# Small perturbation (add small random offset, then clip)
np.random.seed(42)  # for reproducibility
perturbation = np.random.uniform(-0.005, 0.005, size=2)
f1_next = best_point + perturbation
f1_next = np.clip(f1_next, 0, 1)
f1_next = np.round(f1_next, 6)

print(f"\n{'='*60}")
print(f"Function 1 (2D) - Local exploitation (signal found)")
print(f"Best point: {best_point[0]:.6f}-{best_point[1]:.6f} (output: 4.81e-13)")
print(f"Next query (small perturbation): {f1_next[0]:.6f}-{f1_next[1]:.6f}")

# ============================================
# FUNCTION 2 (2D)
# ============================================
X2 = np.array([
    [0.66579958, 0.12396913],
    [0.87779099, 0.77862750],
    [0.14269907, 0.34900513],
    [0.84527543, 0.71112027],
    [0.45464714, 0.29045518],
    [0.57771284, 0.77197318],
    [0.43816606, 0.68501826],
    [0.34174959, 0.02869772],
    [0.33864816, 0.21386725],
    [0.70263656, 0.92656420],
    [0.98803200, 0.66276800],  # Round 1
    [0.72426000, 0.72773700]   # Round 2
])
y2 = np.array([0.53899612, 0.42058624, -0.06562362, 0.29399291, 0.21496451,
               0.02310555, 0.24461934, 0.03874902, -0.01385762, 0.61120522,
               0.09060916, 0.52983021])

f2_next = select_next_query(X2, y2, dim=2, function_name="Function 2")

# ============================================
# FUNCTION 3 (3D)
# ============================================
X3 = np.array([
    [0.17152521, 0.34391687, 0.24873720],
    [0.24211446, 0.64407427, 0.27243281],
    [0.53490572, 0.39850092, 0.17338873],
    [0.49258141, 0.61159319, 0.34017639],
    [0.13462167, 0.21991724, 0.45820622],
    [0.34552327, 0.94135983, 0.26936348],
    [0.15183663, 0.43999062, 0.99088187],
    [0.64550284, 0.39714294, 0.91977134],
    [0.74691195, 0.28419631, 0.22629985],
    [0.17047699, 0.69703240, 0.14916943],
    [0.22054934, 0.29782524, 0.34355534],
    [0.66601366, 0.67198515, 0.24629530],
    [0.04680895, 0.23136024, 0.77061759],
    [0.60009728, 0.72513573, 0.06608864],
    [0.96599485, 0.86111969, 0.56682913],
    [0.51416300, 0.47444700, 0.61889100],  # Round 1
    [0.53042500, 0.65052600, 0.21683200]   # Round 2
])
y3 = np.array([-0.11212220, -0.08796286, -0.11141465, -0.03483531, -0.04800758,
               -0.11062091, -0.39892551, -0.11386851, -0.13146061, -0.09418956,
               -0.04694741, -0.10596504, -0.11804826, -0.03637783, -0.05675837,
               -0.07244085, -0.11634819])

f3_next = select_next_query(X3, y3, dim=3, function_name="Function 3")

# ============================================
# FUNCTION 4 (4D)
# ============================================
X4 = np.array([
    [0.89698105, 0.72562797, 0.17540431, 0.70169437],
    [0.88935640, 0.49958786, 0.53926886, 0.50878344],
    [0.25094624, 0.03369313, 0.14538002, 0.49493242],
    [0.34696206, 0.00625040, 0.76056361, 0.61302356],
    [0.12487118, 0.12977019, 0.38440048, 0.28707610],
    [0.80130271, 0.50023109, 0.70664456, 0.19510284],
    [0.24770826, 0.06044543, 0.04218635, 0.44132425],
    [0.74670224, 0.75709150, 0.36935306, 0.20656628],
    [0.40066503, 0.07257425, 0.88676825, 0.24384229],
    [0.62607060, 0.58675126, 0.43880578, 0.77885769],
    [0.95713529, 0.59764438, 0.76611385, 0.77620991],
    [0.73281243, 0.14524998, 0.47681272, 0.13336573],
    [0.65511548, 0.07239183, 0.68715175, 0.08151656],
    [0.21973443, 0.83203134, 0.48286416, 0.08256923],
    [0.48859419, 0.21196510, 0.93917791, 0.37619173],
    [0.16713049, 0.87655456, 0.21723954, 0.95980098],
    [0.21691119, 0.16608583, 0.24137226, 0.77006248],
    [0.38748784, 0.80453226, 0.75179548, 0.72382744],
    [0.98562189, 0.66693268, 0.15678328, 0.85653480],
    [0.03782483, 0.66485335, 0.16198218, 0.25392378],
    [0.68348638, 0.90277010, 0.33541983, 0.99948256],
    [0.17034731, 0.75695908, 0.27652049, 0.53123150],
    [0.85965692, 0.91959232, 0.20613873, 0.09779683],
    [0.28213837, 0.50598691, 0.53053084, 0.09630162],
    [0.32607578, 0.47236690, 0.45319200, 0.10588734],
    [0.94838936, 0.89451301, 0.85163782, 0.55219629],
    [0.66495539, 0.04656628, 0.11677747, 0.79371778],
    [0.57776561, 0.42877174, 0.42582587, 0.24900741],
    [0.73861301, 0.48210263, 0.70936644, 0.50397001],
    [0.85481080, 0.49396462, 0.73530997, 0.80809201],
    [0.44631900, 0.45519700, 0.38114600, 0.47026800],  # Round 1 
    [0.54549600, 0.48823100, 0.46833100, 0.50728900]   # Round 2
])
y4 = np.array([-22.10828779, -14.60139663, -11.69993246, -16.05376511,
               -10.06963343, -15.48708254, -12.68168498, -16.02639977,
               -17.04923465, -12.74176599, -27.31639636, -13.52764887,
               -16.67911520, -16.50715856, -17.81799934, -26.56182083,
               -12.75832422, -19.44155762, -28.90327367, -13.70274694,
               -29.42709140, -11.56574199, -26.85778644, -7.96677535,
               -6.70208925, -32.62566022, -19.98949793, -4.02554228,
               -13.12278233, -23.13942840, -1.41556207, -4.51863142])

f4_next = select_next_query(X4, y4, dim=4, function_name="Function 4")

# ============================================
# FUNCTION 5 (4D)
# ============================================
X5 = np.array([
    [0.19144708, 0.03819337, 0.60741781, 0.41458414],
    [0.75865295, 0.53651774, 0.65600038, 0.36034155],
    [0.43834987, 0.80433970, 0.21024527, 0.15129482],
    [0.70605083, 0.53419196, 0.26424335, 0.48208755],
    [0.83647799, 0.19360965, 0.66389270, 0.78564888],
    [0.68343225, 0.11866264, 0.82904591, 0.56757661],
    [0.55362148, 0.66734998, 0.32380582, 0.81486975],
    [0.35235627, 0.32224153, 0.11697937, 0.47311252],
    [0.15378571, 0.72938169, 0.42259844, 0.44307417],
    [0.46344227, 0.63002451, 0.10790646, 0.95764390],
    [0.67749115, 0.35850951, 0.47959222, 0.07288048],
    [0.58397341, 0.14724265, 0.34809746, 0.42861465],
    [0.30688872, 0.31687813, 0.62263448, 0.09539906],
    [0.51114177, 0.81795700, 0.72871042, 0.11235362],
    [0.43893338, 0.77409176, 0.37816709, 0.93369621],
    [0.22418902, 0.84648049, 0.87948418, 0.87851568],
    [0.72526172, 0.47987049, 0.08894684, 0.75976022],
    [0.35548161, 0.63961937, 0.41761768, 0.12260384],
    [0.11987923, 0.86254031, 0.64333133, 0.84980383],
    [0.12688467, 0.15342962, 0.77016219, 0.19051811],
    [0.23397200, 0.88247500, 0.86354200, 0.88803100],  # Round 1
    [0.24780600, 0.80435800, 0.71370500, 0.78194000]   # Round 2
])
y5 = np.array([64.4434399, 18.3013796, 0.112939795, 4.21089813,
               258.370525, 78.4343889, 57.5715369, 109.571876,
               8.84799176, 233.223610, 24.4230883, 64.4201468,
               63.4767158, 79.7291299, 355.806818, 1088.85962,
               28.8667516, 45.1815703, 431.612757, 9.97233189,
               1227.07585662, 250.77464727])

f5_next = select_next_query(X5, y5, dim=4, function_name="Function 5")

# ============================================
# FUNCTION 6 (5D)
# ============================================
X6 = np.array([
    [0.72818610, 0.15469257, 0.73255167, 0.69399651, 0.05640131],
    [0.24238435, 0.84409997, 0.57780910, 0.67902128, 0.50195289],
    [0.72952261, 0.74810620, 0.67977464, 0.35655228, 0.67105368],
    [0.77062024, 0.11440374, 0.04677993, 0.64832428, 0.27354905],
    [0.61881230, 0.33180214, 0.18728787, 0.75623847, 0.32883480],
    [0.78495809, 0.91068235, 0.70812010, 0.95922543, 0.00491150],
    [0.14511079, 0.89668460, 0.89632223, 0.72627154, 0.23627199],
    [0.94506907, 0.28845905, 0.97880576, 0.96165559, 0.59801594],
    [0.12572016, 0.86272469, 0.02854433, 0.24660527, 0.75120624],
    [0.75759436, 0.35583141, 0.01652290, 0.43420720, 0.11243304],
    [0.53679690, 0.30878091, 0.41187929, 0.38822518, 0.52252830],
    [0.95773967, 0.23566857, 0.09914585, 0.15680593, 0.07131737],
    [0.62930790, 0.80348368, 0.81140844, 0.04561319, 0.11062446],
    [0.02173531, 0.42808424, 0.83593944, 0.48948866, 0.51108173],
    [0.43934426, 0.69892383, 0.42682022, 0.10947609, 0.87788847],
    [0.25890557, 0.79367771, 0.64211390, 0.19667346, 0.59310318],
    [0.43216593, 0.71561781, 0.34181910, 0.70499988, 0.61496184],
    [0.78287982, 0.53633586, 0.44328356, 0.85969983, 0.01032599],
    [0.92177620, 0.93187122, 0.41487637, 0.59505727, 0.73562569],
    [0.12667892, 0.29147030, 0.06452848, 0.68051460, 0.89281919],
    [0.76902000, 0.17650500, 0.70371200, 0.65735300, 0.05471200],  # Round 1
    [0.67199900, 0.28946300, 0.66867900, 0.70265700, 0.17692100]   # Round 2
])
y6 = np.array([-0.71426495, -1.20995524, -1.67219994, -1.53605771, -0.82923655,
               -1.24704893, -1.23378638, -1.69434344, -2.57116963, -1.30911635,
               -1.14478485, -1.91267714, -1.62283895, -1.35668211, -2.01842540,
               -1.70255784, -1.29424696, -0.93575656, -2.15576776, -1.74688209,
               -0.74900586, -0.47416381])

f6_next = select_next_query(X6, y6, dim=5, function_name="Function 6")

# ============================================
# FUNCTION 7 (6D)
# ============================================
X7 = np.array([
    [0.27262382, 0.32449536, 0.89710881, 0.83295115, 0.15406269, 0.79586362],
    [0.54300258, 0.92469390, 0.34156746, 0.64648585, 0.71844033, 0.34313266],
    [0.09083225, 0.66152938, 0.06593091, 0.25857701, 0.96345285, 0.64026540],
    [0.11886697, 0.61505494, 0.90581639, 0.85530030, 0.41363143, 0.58523563],
    [0.63021764, 0.83809690, 0.68001305, 0.73189509, 0.52673671, 0.34842921],
    [0.76491917, 0.25588292, 0.60908422, 0.21807904, 0.32294277, 0.09579366],
    [0.05789554, 0.49167222, 0.24742222, 0.21811844, 0.42042833, 0.73096984],
    [0.19525188, 0.07922665, 0.55458046, 0.17056682, 0.01494418, 0.10703171],
    [0.64230298, 0.83687455, 0.02179269, 0.10148801, 0.68307083, 0.69241640],
    [0.78994255, 0.19554501, 0.57562333, 0.07365919, 0.25904917, 0.05109986],
    [0.52849733, 0.45742436, 0.36009569, 0.36204551, 0.81689098, 0.63747637],
    [0.72261522, 0.01181284, 0.06364591, 0.16517311, 0.07924415, 0.35995166],
    [0.07566492, 0.33450212, 0.13273274, 0.60831236, 0.91838592, 0.82233079],
    [0.94245084, 0.37743962, 0.48612233, 0.22879108, 0.08263175, 0.71195755],
    [0.14864702, 0.03394336, 0.72880565, 0.31606646, 0.02176938, 0.51691776],
    [0.81711239, 0.54816823, 0.10334758, 0.12436955, 0.72823482, 0.44967361],
    [0.41762629, 0.06409998, 0.24566877, 0.55904080, 0.19153138, 0.25464092],
    [0.72628566, 0.46489581, 0.92457051, 0.80724540, 0.63543840, 0.14341787],
    [0.31981043, 0.52009759, 0.29067775, 0.87670668, 0.49503469, 0.61908250],
    [0.87987128, 0.39796199, 0.00363456, 0.95699064, 0.26451373, 0.11486924],
    [0.54124078, 0.63140314, 0.03190205, 0.44998156, 0.79865282, 0.63370429],
    [0.22634792, 0.11502581, 0.82474966, 0.94538372, 0.90531153, 0.95101392],
    [0.68685257, 0.04101721, 0.00757301, 0.28500900, 0.69156848, 0.65554290],
    [0.17597754, 0.62441650, 0.29554198, 0.46955276, 0.09776977, 0.72814108],
    [0.88164674, 0.20445019, 0.41447436, 0.42038468, 0.26491501, 0.73066019],
    [0.06661051, 0.52804507, 0.81609520, 0.96101714, 0.08650933, 0.77778822],
    [0.93246638, 0.48881189, 0.25860774, 0.95624344, 0.19042781, 0.51985176],
    [0.84686697, 0.14242917, 0.06066859, 0.75629213, 0.55239830, 0.08130609],
    [0.80628208, 0.32412237, 0.72607601, 0.14871213, 0.71937640, 0.36288398],
    [0.47682313, 0.34094195, 0.01433523, 0.88013956, 0.99865470, 0.07966402],
    [0.20918600, 0.59626800, 0.26773000, 0.66644700, 0.98304800, 0.93561500],  # Round 1
    [0.22702100, 0.47759200, 0.32299200, 0.29002700, 0.43770000, 0.73525300]   # Round 2
])
y7 = np.array([0.60443270, 0.56275307, 0.00750324, 0.06142430, 0.27304680,
               0.08374657, 1.36496830, 0.09264495, 0.01786960, 0.03356494,
               0.07351630, 0.20630970, 0.00882563, 0.26840032, 0.61152553,
               0.01479818, 0.27489251, 0.06676325, 0.04211835, 0.00270147,
               0.01820907, 0.00701603, 0.10050661, 0.47539552, 0.67514163,
               0.51645722, 0.00377748, 0.00313433, 0.02134252, 0.09541116,
               0.01087697, 1.58420542])

f7_next = select_next_query(X7, y7, dim=6, function_name="Function 7")

# ============================================
# FUNCTION 8 (8D)
# ============================================
X8 = np.array([
    [0.60499445, 0.29221502, 0.90845275, 0.35550624, 0.20166872, 0.57533801, 0.31031095, 0.73428138],
    [0.17800696, 0.56622265, 0.99486184, 0.21032501, 0.32015266, 0.70790879, 0.63538449, 0.10713163],
    [0.00907698, 0.81162615, 0.52052036, 0.07568668, 0.26511183, 0.09165169, 0.59241515, 0.36732026],
    [0.50602816, 0.65373012, 0.36341078, 0.17798105, 0.09372830, 0.19742533, 0.75582690, 0.29247234],
    [0.35990926, 0.24907568, 0.49599717, 0.70921498, 0.11498719, 0.28920692, 0.55729515, 0.59388173],
    [0.77881834, 0.00341950, 0.33798313, 0.51952778, 0.82090699, 0.53724669, 0.55134710, 0.66003209],
    [0.90864932, 0.06224970, 0.23825955, 0.76660355, 0.13233596, 0.99024381, 0.68806782, 0.74249594],
    [0.58637144, 0.88073573, 0.74502075, 0.54603485, 0.00964888, 0.74899176, 0.23090707, 0.09791562],
    [0.76113733, 0.85467239, 0.38212433, 0.33735198, 0.68970832, 0.30985305, 0.63137968, 0.04195607],
    [0.98493320, 0.69950626, 0.99888550, 0.18014846, 0.58014315, 0.23108719, 0.49082694, 0.31368272],
    [0.11207131, 0.43773566, 0.59659878, 0.59277563, 0.22698177, 0.41010452, 0.92123758, 0.67475276],
    [0.79188751, 0.57619134, 0.69452836, 0.28342378, 0.13675546, 0.27916186, 0.84276726, 0.62532792],
    [0.14355030, 0.93741452, 0.23232482, 0.00904349, 0.41457893, 0.40932517, 0.55377852, 0.20584080],
    [0.76991655, 0.45875909, 0.55900044, 0.69460444, 0.50319902, 0.72834638, 0.78425353, 0.66313109],
    [0.05644741, 0.06595555, 0.02292868, 0.03878647, 0.40393544, 0.80105533, 0.48830701, 0.89308498],
    [0.86243745, 0.48273382, 0.28186940, 0.54410223, 0.88749026, 0.38265469, 0.60190199, 0.47646169],
    [0.35151190, 0.59006494, 0.90943630, 0.67840835, 0.21282566, 0.08846038, 0.41015300, 0.19572429],
    [0.73590364, 0.03461189, 0.72803027, 0.14742652, 0.29574314, 0.44511731, 0.97517969, 0.37433978],
    [0.68029397, 0.25510465, 0.86218799, 0.13439582, 0.32632920, 0.28790687, 0.43501048, 0.36420013],
    [0.04432925, 0.01358149, 0.25819824, 0.57764416, 0.05127992, 0.15856307, 0.59103012, 0.07795293],
    [0.77834548, 0.75114565, 0.31414221, 0.90298577, 0.33538166, 0.38632267, 0.74897249, 0.98875510],
    [0.89888711, 0.52364170, 0.87678325, 0.21869645, 0.90026089, 0.28276624, 0.91107791, 0.47239822],
    [0.14512029, 0.11932754, 0.42088822, 0.38760861, 0.15542283, 0.87517163, 0.51055967, 0.72861058],
    [0.33895442, 0.56693202, 0.37675110, 0.09891573, 0.65945169, 0.24554809, 0.76248278, 0.73215347],
    [0.17615002, 0.29396143, 0.97567997, 0.79393631, 0.92340076, 0.03084229, 0.80325452, 0.59589758],
    [0.02894663, 0.02827906, 0.48137155, 0.61317460, 0.67266045, 0.02211341, 0.60148330, 0.52488505],
    [0.19263987, 0.63067728, 0.41679584, 0.49052929, 0.79608602, 0.65456706, 0.27624119, 0.29551759],
    [0.94318502, 0.21885062, 0.72118408, 0.42459707, 0.98690200, 0.53518298, 0.71474318, 0.96009372],
    [0.53272140, 0.83369260, 0.07139900, 0.11681148, 0.73069311, 0.93737559, 0.86650798, 0.12790200],
    [0.44709584, 0.84395253, 0.72954612, 0.63915138, 0.40928714, 0.13264569, 0.03590888, 0.44683847],
    [0.38222497, 0.55713584, 0.85310163, 0.33379569, 0.26572127, 0.48087292, 0.23764706, 0.76863196],
    [0.53281953, 0.86230848, 0.53826712, 0.04944293, 0.71970119, 0.90670590, 0.10823094, 0.52534791],
    [0.39486519, 0.33180167, 0.74075430, 0.69786172, 0.73740444, 0.78377681, 0.25449546, 0.87114551],
    [0.98594539, 0.87305363, 0.07039262, 0.05358729, 0.73415296, 0.52025852, 0.81104004, 0.10336036],
    [0.96457339, 0.97397979, 0.66375335, 0.66221599, 0.67312167, 0.90523762, 0.45887462, 0.56091750],
    [0.47207071, 0.16820264, 0.08642757, 0.45265551, 0.48061922, 0.62243949, 0.92897446, 0.11253627],
    [0.85600695, 0.63889370, 0.32619202, 0.66850311, 0.24029837, 0.21029889, 0.16754636, 0.96358986],
    [0.81003174, 0.63504604, 0.26954758, 0.86960534, 0.66192159, 0.25225873, 0.76567003, 0.89054867],
    [0.79625252, 0.00703653, 0.35569738, 0.48756605, 0.74051962, 0.70665010, 0.99291449, 0.38173437],
    [0.48124533, 0.10246072, 0.21948594, 0.67732237, 0.24750919, 0.24434086, 0.16382453, 0.71596164],
    [0.12723600, 0.19947000, 0.06743700, 0.24756600, 0.85699100, 0.31539100, 0.05097700, 0.60213000],  # Round 1
    [0.12031000, 0.20045300, 0.10541500, 0.24192100, 0.75375200, 0.44129700, 0.13801700, 0.66839000]   # Round 2
])
y8 = np.array([7.39872110, 7.00522736, 8.45948162, 8.28400781, 8.60611679,
               8.54174792, 7.32743458, 7.29987205, 7.95787474, 5.59219339,
               7.85454099, 6.79198578, 8.97655402, 7.37908290, 9.59848200,
               8.15998319, 7.13162397, 6.76796253, 7.43374407, 9.01307515,
               7.31089382, 5.84106731, 9.14163949, 8.81755844, 6.45194313,
               8.83074505, 9.34427428, 6.88784639, 8.04221254, 7.69236805,
               7.92375877, 8.42175924, 8.27806240, 7.11345716, 6.40258841,
               8.47293632, 7.97768459, 7.46087219, 7.43659353, 9.18300525,
               9.89468698, 9.97369978])

f8_next = select_next_query(X8, y8, dim=8, function_name="Function 8")

# ============================================
# FINAL OUTPUT - COPY THESE INTO PORTAL
# ============================================
print("\n" + "="*60)
print("FINAL QUERIES FOR SUBMISSION (ROUND 3)")
print("="*60)
print(f"\nFunction 1 (2D): {f1_next[0]:.6f}-{f1_next[1]:.6f}")
print(f"Function 2 (2D): {f2_next[0]:.6f}-{f2_next[1]:.6f}")
print(f"Function 3 (3D): {f3_next[0]:.6f}-{f3_next[1]:.6f}-{f3_next[2]:.6f}")
print(f"Function 4 (4D): {f4_next[0]:.6f}-{f4_next[1]:.6f}-{f4_next[2]:.6f}-{f4_next[3]:.6f}")
print(f"Function 5 (4D): {f5_next[0]:.6f}-{f5_next[1]:.6f}-{f5_next[2]:.6f}-{f5_next[3]:.6f}")
print(f"Function 6 (5D): {f6_next[0]:.6f}-{f6_next[1]:.6f}-{f6_next[2]:.6f}-{f6_next[3]:.6f}-{f6_next[4]:.6f}")
print(f"Function 7 (6D): {f7_next[0]:.6f}-{f7_next[1]:.6f}-{f7_next[2]:.6f}-{f7_next[3]:.6f}-{f7_next[4]:.6f}-{f7_next[5]:.6f}")
print(f"Function 8 (8D): {f8_next[0]:.6f}-{f8_next[1]:.6f}-{f8_next[2]:.6f}-{f8_next[3]:.6f}-{f8_next[4]:.6f}-{f8_next[5]:.6f}-{f8_next[6]:.6f}-{f8_next[7]:.6f}")


Function 1 (2D) - Local exploitation (signal found)
Best point: 0.445446-0.303395 (output: 4.81e-13)
Next query (small perturbation): 0.444191-0.307902

  Function 2:
  Baseline LOOCV score: -0.0512
  Current best: 0.6112 at [0.70263656 0.9265642 ]

  Logistic Regression Diagnostic (threshold=median):
    Accuracy: 0.6667
    Coefficients (direction: + = higher Y, - = lower Y):
      x1: +0.5618 ↑
      x2: +0.5178 ↑

    Tuning Ridge...
      Ridge: LOOCV=-0.0684, beats_baseline=False, at_boundary=False
      Suggestion: [0.833267 0.809557]

    Tuning KNN...
      KNN: LOOCV=-0.0509, beats_baseline=True, at_boundary=False
      Suggestion: [0.825136 0.556073]

    Tuning RandomForest...
      RandomForest: LOOCV=-0.0445, beats_baseline=True, at_boundary=False
      Suggestion: [0.612911 0.742973]

    Tuning SVR...
      SVR: LOOCV=-0.0482, beats_baseline=True, at_boundary=False
      Suggestion: [0.591487 0.806336]

    Tuning GP...
      GP: LOOCV=-0.1107, beats_baseline=False, at